# Customer Analytics, Segmentation & Revenue Forecasting

**Dataset:** UCI Machine Learning Repository — Online Retail II  
**Business objective:** turn transaction data into customer segments, revenue trends, and a short-horizon revenue forecast.

### Questions answered
1. What data-quality issues must be handled before analysis?
2. Which markets/products/customers drive revenue?
3. Can customers be segmented into behaviorally distinct groups using RFM features?
4. Can a classical forecasting model beat a simple seasonal baseline on held-out future data?


In [ ]:
!pip -q install ucimlrepo statsmodels scikit-learn duckdb openpyxl


In [ ]:
import io
import zipfile
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import duckdb


## 1. Load the UCI Online Retail II dataset


In [ ]:
# The ucimlrepo wrapper may not expose dataset 502 directly.
# Fallback: download the official UCI ZIP in memory and load both Excel sheets.
try:
    online_retail_ii = fetch_ucirepo(id=502)
    raw = online_retail_ii.data.features.copy()
except Exception:
    url = 'https://archive.ics.uci.edu/static/public/502/online%2Bretail%2Bii.zip'
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        workbooks = [name for name in archive.namelist() if name.lower().endswith('.xlsx')]
        if not workbooks:
            raise RuntimeError('No Excel workbook found in the official UCI ZIP.')
        workbook_bytes = io.BytesIO(archive.read(workbooks[0]))
    sheets = pd.read_excel(workbook_bytes, sheet_name=None)
    raw = pd.concat(sheets.values(), ignore_index=True)
print('Raw shape:', raw.shape)
display(raw.head())


## 2. Standardize schema and audit the raw data


In [ ]:
def canonicalize_columns(df):
    normalized = {str(c).strip().lower().replace('_','').replace(' ',''): c for c in df.columns}
    variants = {
        'invoice':['invoice','invoiceno'], 'stock_code':['stockcode'],
        'description':['description'], 'quantity':['quantity'],
        'invoice_date':['invoicedate'], 'unit_price':['unitprice','price'],
        'customer_id':['customerid','customer'], 'country':['country']}
    rename = {}
    for canonical, keys in variants.items():
        for k in keys:
            if k in normalized:
                rename[normalized[k]] = canonical
                break
    result = df.rename(columns=rename).copy()
    required = ['invoice','stock_code','description','quantity','invoice_date','unit_price','customer_id','country']
    return result[required]
df = canonicalize_columns(raw)
print(df.shape)
display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:', df.duplicated().sum())
df['is_cancellation'] = df['invoice'].astype(str).str.strip().str.upper().str.startswith('C')
df['revenue_raw'] = df['quantity'] * df['unit_price']
print('Quantity <= 0:', (df['quantity'] <= 0).sum())
print('Unit price <= 0:', (df['unit_price'] <= 0).sum())
print('Cancellation rows:', df['is_cancellation'].sum())


## 3. Clean for positive realized-revenue analysis


In [ ]:
clean = df.copy()
clean['invoice_date'] = pd.to_datetime(clean['invoice_date'], errors='coerce')
clean = clean.drop_duplicates()
clean = clean[(~clean['is_cancellation']) & (clean['quantity'] > 0) & (clean['unit_price'] > 0) & clean['invoice_date'].notna()].copy()
clean['revenue'] = clean['quantity'] * clean['unit_price']
transactions = clean.copy()
customer_txn = clean[clean['customer_id'].notna()].copy()
customer_txn['customer_id'] = customer_txn['customer_id'].astype(str)
print('Raw rows:', len(df))
print('Positive realized-revenue rows:', len(transactions))
print('Customer-attributable rows:', len(customer_txn))


## 4. Business EDA and SQL


In [ ]:
monthly = transactions.set_index('invoice_date')['revenue'].resample('MS').sum().reset_index()
plt.figure(figsize=(11,4))
plt.plot(monthly['invoice_date'], monthly['revenue'], marker='o')
plt.title('Monthly Positive Realized Revenue')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
transactions['invoice'] = transactions['invoice'].astype(str)
duckdb.register('transactions', transactions)
display(duckdb.sql("""SELECT DATE_TRUNC('month', invoice_date) AS month, ROUND(SUM(revenue),2) AS revenue, COUNT(DISTINCT invoice) AS orders FROM transactions GROUP BY 1 ORDER BY 1""").df().tail())
display(duckdb.sql("""SELECT country, ROUND(SUM(revenue),2) AS revenue, COUNT(DISTINCT invoice) AS orders FROM transactions GROUP BY country ORDER BY revenue DESC LIMIT 10""").df())


## 5. RFM customer segmentation with K-Means


In [ ]:
snapshot_date = customer_txn['invoice_date'].max().normalize() + pd.Timedelta(days=1)
rfm = customer_txn.groupby('customer_id', as_index=False).agg(last_purchase=('invoice_date','max'), frequency=('invoice','nunique'), monetary=('revenue','sum'))
rfm['recency'] = (snapshot_date - rfm['last_purchase'].dt.normalize()).dt.days
rfm = rfm[(rfm['frequency'] > 0) & (rfm['monetary'] > 0)].copy()
features_log = np.log1p(rfm[['recency','frequency','monetary']])
X_scaled = StandardScaler().fit_transform(features_log)
scores, models = {}, {}
for k in range(2,7):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    models[k] = km
best_k = max(scores, key=scores.get)
rfm['cluster'] = models[best_k].predict(X_scaled)
cluster_profile = rfm.groupby('cluster', as_index=False).agg(customers=('customer_id','size'), avg_recency=('recency','mean'), avg_frequency=('frequency','mean'), avg_monetary=('monetary','mean'), total_monetary=('monetary','sum'))
cluster_profile['customer_share_pct'] = cluster_profile['customers'] / cluster_profile['customers'].sum() * 100
display(cluster_profile.sort_values('avg_monetary', ascending=False))
print('Selected K:', best_k, 'Silhouette:', round(scores[best_k],3))


## 6. Daily revenue forecasting — seasonal naive vs Holt-Winters


In [ ]:
daily = transactions.set_index('invoice_date')['revenue'].resample('D').sum()
daily = daily.reindex(pd.date_range(daily.index.min(), daily.index.max(), freq='D'), fill_value=0.0)
test_days = 28
train, test = daily.iloc[:-test_days], daily.iloc[-test_days:]
naive_pred = np.resize(train.iloc[-7:].to_numpy(), len(test))
hw = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=7, initialization_method='estimated').fit(optimized=True)
hw_pred = hw.forecast(len(test)).to_numpy()
def metrics(actual, pred):
    return {'MAE': mean_absolute_error(actual,pred), 'RMSE': mean_squared_error(actual,pred)**0.5}
forecast_comparison = pd.DataFrame([{'model':'Seasonal naive', **metrics(test.to_numpy(), naive_pred)}, {'model':'Holt-Winters', **metrics(test.to_numpy(), hw_pred)}]).sort_values('MAE')
display(forecast_comparison)


## 7. Export dashboard inputs and verified project facts


In [ ]:
forecast_results = pd.DataFrame({'date':test.index, 'actual':test.to_numpy(), 'seasonal_naive':naive_pred, 'holt_winters':hw_pred})
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)
rfm.to_csv(output_dir / 'customer_segments.csv', index=False)
cluster_profile.to_csv(output_dir / 'cluster_profile.csv', index=False)
forecast_results.to_csv(output_dir / 'forecast_results.csv', index=False)
forecast_comparison.to_csv(output_dir / 'forecast_comparison.csv', index=False)
best, other = forecast_comparison.iloc[0], forecast_comparison.iloc[-1]
print('=== VERIFIED PROJECT FACTS ===')
print(f'Raw transactions: {len(df):,}')
print(f'Clean positive realized-revenue transactions: {len(transactions):,}')
print(f'Customer-attributable transactions: {len(customer_txn):,}')
print(f'Unique customers used for RFM: {rfm.customer_id.nunique():,}')
print(f'K selected by silhouette among 2-6: {best_k}')
print(f'Silhouette score: {scores[best_k]:.3f}')
print(f'Best forecast model by test MAE: {best.model}')
print(f'Best test MAE: £{best.MAE:,.2f}')
print(f'Best test RMSE: £{best.RMSE:,.2f}')
print(f'Relative MAE improvement of best vs other model: {(other.MAE-best.MAE)/other.MAE*100:.1f}%')
